# Bolbosh KTTS v19 evaluation
Generates matched baseline/fine-tuned samples for human review.


In [ ]:
!git clone -q --depth 1 https://github.com/gaash-lab/Bolbosh.git /kaggle/working/Bolbosh
%cd /kaggle/working/Bolbosh
!pip install -q 'numpy<2' Cython soundfile scipy huggingface_hub lightning hydra-core hydra-colorlog rootutils rich torchmetrics librosa einops inflect Unidecode conformer diffusers peft wget
!pip install -q git+https://github.com/abdulmuizz0903/KashmiriNormalizer
!python -c "from setuptools import setup,Extension; from Cython.Build import cythonize; import numpy; setup(name='matcha-align',ext_modules=cythonize([Extension('matcha.utils.monotonic_align.core',['matcha/utils/monotonic_align/core.pyx'])],language_level=3),include_dirs=[numpy.get_include()])" build_ext --inplace
from pathlib import Path
cli=Path('/kaggle/working/Bolbosh/matcha/cli.py')
source=cli.read_text()
source=source.replace('torch.load(checkpoint_path, map_location=device)["generator"]', 'torch.load(checkpoint_path, map_location=device, weights_only=False)["generator"]')
source=source.replace('MatchaTTS.load_from_checkpoint(checkpoint_path, map_location=device)', 'MatchaTTS.load_from_checkpoint(checkpoint_path, map_location=device, weights_only=False)')
cli.write_text(source)


In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download
import subprocess, sys
fine = next(Path('/kaggle/input').rglob('kashmiri-bolbosh-ktts-v1.zip'))
subprocess.run(['unzip','-o',str(fine),'model.ckpt','-d','/kaggle/working/fine'], check=True)
base = hf_hub_download('GAASH-Lab/Matcha-TTS-Kashmiri', 'model.ckpt')
texts = ['السلام علیکم، تُہۍ کِتھ پٲٹھۍ چھِو؟', 'مےٚ ناو چھُ سمیر، بیٚہ چھُس کویل پُلوامہ پٮ۪ٹھ۔', 'اَسہِ چھِ کٲشُر زبان واریاہ پیٛارِ۔']
Path('/kaggle/working/sentences.txt').write_text('\n'.join(texts), encoding='utf-8')
for name, ckpt in [('baseline', base), ('finetuned', '/kaggle/working/fine/model.ckpt')]:
    cmd=[sys.executable,'-m','matcha.cli','--checkpoint_path',str(ckpt),'--vocoder','hifigan_univ_v1','--file','/kaggle/working/sentences.txt','--output_folder',f'/kaggle/working/evaluation/{name}','--steps','10','--speaking_rate','1.0','--spk','0']
    subprocess.run(cmd, cwd='/kaggle/working/Bolbosh', check=True)


In [ ]:
import json, numpy as np, soundfile as sf
from pathlib import Path
rows=[]
for wav in sorted(Path('/kaggle/working/evaluation').rglob('*.wav')):
    y,sr=sf.read(wav); peak=float(np.max(np.abs(y))); rms=float(np.sqrt(np.mean(y*y)))
    rows.append({'model':wav.parent.name,'file':wav.name,'seconds':round(len(y)/sr,3),'sampleRate':sr,'peak':round(peak,5),'rms':round(rms,5),'clippedSamples':int(np.sum(np.abs(y)>=0.999))})
report={'sentences':texts,'measurements':rows,'decision':'human listening review required before deployment'}
Path('/kaggle/working/evaluation/report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(report,ensure_ascii=False,indent=2))
!cd /kaggle/working && zip -qr ktts-v19-evaluation.zip evaluation sentences.txt
